# A decaying activation score

A running score remembers recent strong stimuli but fades with time. At each step the new score
is the largest of three numbers: the score one step back less 2, the score two steps back less
1, and the stimulus of the current step. The older score fades more slowly, so the score depends
on the last two steps and not only on the last one. Before the first step every earlier score is
-1e9, which is lower than any stimulus. Each score depends on the two before it, so the loop
below cannot be replaced by operations on whole arrays.

In [ ]:
import numpy as np

NO_SCORE = -1.0e9  # the value of an earlier score that does not exist


def decay(w):
    """The score at each step: out[i] = max(out[i-1] - 2, out[i-2] - 1, w[i]).

    A score before the first step is NO_SCORE.
    """
    out = np.empty_like(w)
    last, before_last = NO_SCORE, NO_SCORE
    for i in range(len(w)):
        out[i] = max(last - 2.0, before_last - 1.0, w[i])
        before_last, last = last, out[i]
    return out

In [ ]:
print(decay(np.array([5.0, 0.0, 0.0, 0.0, 9.0, 0.0, 0.0])))

The declaration below tells the harness how to call the function and how to check a routine
against it. The stimuli and scores are whole numbers well below 2 to the power 53, so a double
precision routine computes them exactly, in any order of working, and the tolerance is zero.
`make_inputs` draws whole-number stimuli from a range whose top is 2, 10, 1000 or a million,
chosen at random for each call, so that the score is sometimes set by the last few steps and
sometimes by a stimulus long past. A few stimuli are set to the lowest or the highest value of
the range.

In [ ]:
def make_inputs(size, rng):
    high = int(rng.choice([2, 10, 1_000, 1_000_000]))
    w = rng.integers(0, high, size).astype(np.float64)
    special = rng.random(size) < 0.05
    w[special] = rng.choice([0.0, high - 1.0], special.sum())
    return {"w": w}


ROUTINE = {
    "function": decay,
    "make_inputs": make_inputs,
    "arguments": {"w": "float64[n]"},
    "results": {"out": "float64[n]"},
    "test_sizes": [1, 1_000, 100_000],
    "benchmark_size": 60_000_000,
    "tolerance": 0,
}